# CIS6008 Task C — SAM3 Colab GPU — BIA Aerial Segmentation

**Raster:** `BIA_georeferenced_EPSG5234.tif` 8205×4000 EPSG:5234 0.81 m/px  
**Prompts:** `building`, `tree`, `vegetation`, `road` → separate GPKGs  
**Device:** Colab T4/L4 GPU (local M4 was CPU-only, 10.7s/800×800 vit_b, not class-specific — see `99_TEMP/samgeo_test/SAMGEO_TEST_REPORT.md`)

> Upload the FULL `BIA_georeferenced_EPSG5234.tif` (93.9 MB) for radar PSR 3km compliance, or the CORE clip `BIA_CORE_500m_EPSG5234.tif` (39.6 MB) for a quick 5-min test. Do NOT overwrite `03_Original_Datasets/`.

Workflow: **Colab GPU SAM3 (candidate) → download GPKG → local QGIS manual QA → qgis_process buffers/intersections**  
Statement for report: *Automated segmentation was used to generate candidate vector features from the georeferenced aerial imagery, followed by visual verification and manual correction in QGIS.*

## Cell 1 — Install + GPU check (run once, ~2-3 min first time, checkpoint ~2 GB cached)

In [ ]:
!nvidia-smi
import sys, torch
print(f"python {sys.version}")
print(f"torch {torch.__version__} cuda={torch.cuda.is_available()} device_count={torch.cuda.device_count() if torch.cuda.is_available() else 0}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

# Install SAM3 stack (meta backend needs sam3, transformers for HF)
!pip -q install "segment-geospatial[samgeo3]" --no-input
print("install done — restart runtime if first install prompts restart, then re-run from Cell 2")

## Cell 1b — Hugging Face login (REQUIRED, SAM3 is gated)
Facebook SAM3 is a **gated repo** — you must request access and authenticate, otherwise you get `401 Unauthorized / GatedRepoError`.
1. Open https://huggingface.co/facebook/sam3 → click **Agree and access repository** (and https://huggingface.co/facebook/sam3.1 if shown)
2. Create a token at https://huggingface.co/settings/tokens → New token → **Read** type → Copy `hf_...`
3. Paste below and run. After login, re-run Cell 3.
If you already have access, this cell will say `Login successful`.


In [ ]:
# --- Cell 1b: HF login ---
!pip -q install -U huggingface_hub  # ensure latest
from huggingface_hub import login, whoami
import os

# PASTE YOUR TOKEN HERE (keep the quotes). Example: HF_TOKEN = "hf_xxxxxxxxxxxxxxxx"
HF_TOKEN = ""  # <-- paste hf_... here, then run this cell

if not HF_TOKEN:
    print("⚠️  Paste your hf_ token into HF_TOKEN and re-run.")
    print("Get token at https://huggingface.co/settings/tokens → New token (Read)")
    print("Request access at https://huggingface.co/facebook/sam3 → Agree and access repository")
    print("Also request https://huggingface.co/facebook/sam3.1 if you use sam3.1")
else:
    try:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("Login successful")
        # also set env for subprocesses
        os.environ["HF_TOKEN"] = HF_TOKEN
        os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
        # verify access (lightweight HEAD)
        from huggingface_hub import hf_hub_download
        try:
            hf_hub_download(repo_id="facebook/sam3", filename="config.json", token=HF_TOKEN)
            print("Verified: can access facebook/sam3 (config.json)")
        except Exception as e:
            print(f"Still cannot access facebook/sam3: {e}")
            print("Did you click Agree on https://huggingface.co/facebook/sam3 ? Wait 1-2 min and retry.")
    except Exception as e:
        print(f"login failed: {e}")


## Cell 2 — Upload raster
Use Files pane (left) → Upload `BIA_georeferenced_EPSG5234.tif` (or `BIA_CORE_500m_EPSG5234.tif`).  
Or mount Drive:
```python
from google.colab import drive; drive.mount('/content/drive')
SOURCE = "/content/drive/MyDrive/BIA_georeferenced_EPSG5234.tif"
```

In [ ]:
import os, pathlib, rasterio
from google.colab import files

# --- SET THIS after upload ---
SOURCE = "/content/BIA_georeferenced_EPSG5234.tif"  # or "/content/BIA_CORE_500m_EPSG5234.tif"

# If you haven't uploaded yet, this will prompt:
if not os.path.exists(SOURCE):
    print(f"{SOURCE} not found — please upload via Files pane (drag & drop)")
    print("Or run: uploaded = files.upload() and move it")
    # uploaded = files.upload()
else:
    with rasterio.open(SOURCE) as src:
        print(f"SOURCE: {SOURCE}")
        print(f"  size {src.width}x{src.height} {src.count} bands {src.dtypes}")
        print(f"  CRS {src.crs} transform {src.transform}")
        print(f"  bounds {src.bounds}")
        print(f"  file {os.path.getsize(SOURCE)/1e6:.1f} MB")
    # List files
    print("\n/content:", os.listdir("/content"))

## Cell 3 — SAM3 tiled segmentation (4 prompts)
Tiled = avoids 32.8 MP OOM. 1024×1024 +128 overlap is SAM3 default for large GeoTIFFs.  
Each prompt takes ~3-6 min on T4 for FULL raster; CORE ~1-2 min each.  
Outputs: `*_mask.tif` (uint32 unique IDs, EPSG:5234) — then vectorized to `*.gpkg` in Cell 4.

In [ ]:
import os, time, pathlib
from samgeo import SamGeo3
import torch

SOURCE = "/content/BIA_georeferenced_EPSG5234.tif"  # change if using CORE
OUT_DIR = "/content/sam3_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

PROMPTS = ["building", "tree", "vegetation", "road"]
# If road is poor, try alt prompts: "asphalt road", "paved road"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={device} backend=meta model=facebook/sam3.1")

# SAM3 Meta backend — downloads facebook/sam3.1 checkpoint on first run (~2 GB)
sam = SamGeo3(backend="meta", device=device, resolution=1008, confidence_threshold=0.5)
# NOTE: requires HF login above (gated repo). If 401 persists, ensure you clicked Agree on HF model page.
print(f"model_id={sam.model_id} device={sam.device}")

for prompt in PROMPTS:
    out_tif = os.path.join(OUT_DIR, f"{prompt}_mask.tif")
    print("\n" + "="*70)
    print(f"PROMPT: {prompt} → {out_tif}")
    print("="*70)
    t0 = time.time()
    # tiled: tile_size 1024, overlap 128 — tune if OOM (768/192)
    sam.generate_masks_tiled(
        source=SOURCE,
        prompt=prompt,
        output=out_tif,
        tile_size=1024,
        overlap=128,
        min_size=0,  # filter after vectorizing
        unique=True,
        dtype="uint32",
        verbose=True,
    )
    dt = time.time() - t0
    print(f"done {prompt} in {dt:.1f}s → {out_tif} ({os.path.getsize(out_tif)/1e6:.1f} MB)")
    # Free GPU cache between prompts
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nAll prompts done. Outputs:", os.listdir(OUT_DIR))
!ls -lh /content/sam3_outputs/

## Cell 4 — Vectorize to GPKG + preview overlays
Raster mask (unique IDs) → vector polygons via `raster_to_gpkg`. Preserves EPSG:5234 georeferencing.  
Check counts + quick matplotlib overlay before downloading.

In [ ]:
import os, glob
from samgeo.common import raster_to_gpkg
import geopandas as gpd
import rasterio, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

OUT_DIR = "/content/sam3_outputs"
SOURCE = "/content/BIA_georeferenced_EPSG5234.tif"

for prompt in ["building","tree","vegetation","road"]:
    tif = os.path.join(OUT_DIR, f"{prompt}_mask.tif")
    gpkg = os.path.join(OUT_DIR, f"{prompt}.gpkg")
    if not os.path.exists(tif):
        print(f"skip {prompt}: {tif} not found")
        continue
    print(f"\nvectorize {tif} → {gpkg}")
    raster_to_gpkg(tif, gpkg, simplify_tolerance=1.0)  # 1.0 m = ~1 px simplify
    # stats
    try:
        gdf = gpd.read_file(gpkg)
        print(f"  {prompt}: {len(gdf)} features CRS={gdf.crs} size {os.path.getsize(gpkg)/1e6:.2f} MB")
        if len(gdf)>0:
            gdf["area_m2"] = gdf.geometry.area
            print(gdf["area_m2"].describe().to_string())
    except Exception as e:
        print(f"  read fail {e}")

print("\ngpkgs:", glob.glob(os.path.join(OUT_DIR, "*.gpkg")))
print("tifs:", glob.glob(os.path.join(OUT_DIR, "*_mask.tif")))

# Quick overlay for building (first 5000×5000 px window would be heavy, so plot full extent with decimation)
# Instead plot a 1500×1200 window around Tower (102030,219787) for visual check
try:
    import rasterio.windows
    tower_x, tower_y = 102030, 219787
    px = 0.8133757
    w, h = 1500, 1200
    ulx, uly = tower_x - w*px/2, tower_y + h*px/2
    lrx, lry = tower_x + w*px/2, tower_y - h*px/2
    with rasterio.open(SOURCE) as src:
        win = rasterio.windows.from_bounds(ulx, lry, lrx, uly, transform=src.transform)
        win = win.intersection(rasterio.windows.Window(0,0,src.width,src.height))
        arr = np.transpose(src.read(window=win), (1,2,0))
        bounds = rasterio.windows.bounds(win, transform=src.transform)
        fig, axes = plt.subplots(2,2, figsize=(14,10))
        axes = axes.flatten()
        for i, prompt in enumerate(["building","tree","vegetation","road"]):
            ax = axes[i]
            ax.imshow(arr, extent=[bounds[0],bounds[2],bounds[1],bounds[3]], origin='upper')
            gpkg = os.path.join(OUT_DIR, f"{prompt}.gpkg")
            if os.path.exists(gpkg):
                gdf = gpd.read_file(gpkg)
                # clip to window for plot speed
                from shapely.geometry import box as sbox
                wbox = sbox(*bounds)
                gdf_c = gdf[gdf.geometry.intersects(wbox)]
                if len(gdf_c)>0:
                    gdf_c.boundary.plot(ax=ax, color="red", linewidth=0.6, alpha=0.7)
                ax.set_title(f"{prompt} — {len(gdf)} total, {len(gdf_c)} in window", fontsize=9)
            else:
                ax.set_title(f"{prompt} — no gpkg", fontsize=9)
            ax.plot(tower_x, tower_y, 'bo', markersize=6, markeredgecolor='yellow')
            ax.set_xlabel("Easting")
            ax.set_ylabel("Northing")
        plt.tight_layout()
        out_png = os.path.join(OUT_DIR, "preview_4prompts_tower.png")
        plt.savefig(out_png, dpi=200)
        print(f"preview saved {out_png}")
        from IPython.display import Image, display
        display(Image(filename=out_png))
except Exception as e:
    import traceback; traceback.print_exc()
    print(f"preview fail {e}")

## Cell 5 — Download (or zip)
Download via Files pane, or run the zip command and then download the zip.

In [ ]:
!ls -lh /content/sam3_outputs/
!zip -r /content/sam3_outputs.zip /content/sam3_outputs/*.gpkg /content/sam3_outputs/*_mask.tif /content/sam3_outputs/*.png 2>&1 | tail -n 20
print("\nDownload /content/sam3_outputs.zip via Files pane (right-click → Download)")
print("Or download individual .gpkg files")
# Alternative: files.download per file (uncomment if needed)
# from google.colab import files
# for f in glob.glob("/content/sam3_outputs/*.gpkg"):
#     files.download(f)

## After Colab — local steps
1. Move downloaded `*.gpkg` to `06_Task_C_QGIS/digitized_layers/colab_raw/`
2. Run post-processing:
```bash
 .venv/bin/python 06_Task_C_QGIS/colab/post_process_colab_outputs.py
```
3. QGIS manual QA (see `COLAB_SAM3_GUIDE.md` §4) — delete false positives, reshape roofs.
4. Then `qgis_process` buffers/intersections + PostGIS.

Do NOT claim AI output was hand-digitised. See `COLAB_SAM3_GUIDE.md` for report statement.
